In [1]:
##### Bibliotecas
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt

# Ignite
from ignite.engine import Engine, Events
from ignite.handlers import EarlyStopping
from ignite.metrics import Accuracy, Loss

# Optuna
import optuna

# Organização do dataset
data = "/home/jovyan/DADOS-DIVIDIDOS"
feature_extract = True

In [2]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
        transforms.RandomVerticalFlip(),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.5, interpolation=3, fill=0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

image_datasets = {x: datasets.ImageFolder(os.path.join(data, x), data_transforms[x]) for x in ['train', 'val', 'test']}

/opt/conda/lib/python3.10/site-packages/torchvision/transforms/transforms.py:768: UserWarning: Argument 'interpolation' of type int is deprecated since 0.13 and will be removed in 0.15. Please use InterpolationMode enum.
  warnings.warn(


In [3]:
# Extração de features + Congelamento dos parâmetros
def set_parameter_requires_grad(model, feature_extracting):
    if feature_extracting:
        for param in model.parameters():
            param.requires_grad = False
            
            
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.vgg19(pretrained=False)
set_parameter_requires_grad(model, feature_extract)

model_vgg19 = "/home/jovyan/models/VggNet19-model-96.pth"
state_dict = torch.load(model_vgg19)

del state_dict['classifier.6.weight']
del state_dict['classifier.6.bias']

model.load_state_dict(state_dict, strict=False)

model.to(device)



/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padd

In [ ]:
# Função de treinamento e validação
def train_step(engine, batch):
    x, y = batch
    x, y = x.to(device), y.to(device)

    model.train()
    y_pred = model(x)
    loss = criterion(y_pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

def validation_step(engine, batch):
    model.eval()
    with torch.no_grad():
        x, y = batch
        x, y = x.to(device), y.to(device)
        y_pred = model(x)
        return y_pred, y

# Função `objective` para busca bayesiana com Optuna
def objective(trial):
    global model, optimizer, criterion
    
    # Hiperparâmetros
    dropout_rate1 = trial.suggest_uniform("dropout1", 0.2, 0.5)
    dropout_rate2 = trial.suggest_uniform("dropout2", 0.2, 0.5)
    activation = trial.suggest_categorical("activation", ["ReLU"])
    batch_size = trial.suggest_categorical("batch_size", [128])
    num_neurons_in = trial.suggest_categorical("num_neurons_in", [256, 512, 1024, 4096])
    num_neurons_out = trial.suggest_categorical("num_neurons_out", [256, 512, 1024, 4096])
    optimizer_name = trial.suggest_categorical("optimizer", ["SGD", "Adam"])
    lr = trial.suggest_loguniform("lr", 1e-5, 1e-2)
    momentum = trial.suggest_uniform("momentum", 0.7, 0.99) if optimizer_name == "SGD" else None

    
    # Função de ativação
    activation_fn = getattr(nn, activation)()

    # Redefinir o DataLoader com o batch size sugerido
    dataloaders_dict = {
        'train': torch.utils.data.DataLoader(image_datasets['train'], batch_size=batch_size, shuffle=True),
        'val': torch.utils.data.DataLoader(image_datasets['val'], batch_size=batch_size, shuffle=False)
    }

    # Configurar o modelo com os hiperparâmetros sugeridos
    model.classifier = nn.Sequential(
        nn.Linear(in_features=25088, out_features=num_neurons_out, bias=True),
        nn.ReLU(inplace=True),
        nn.Dropout(p=dropout_rate1, inplace=False),
        nn.Linear(in_features=num_neurons_out, out_features=num_neurons_in, bias=True),
        nn.ReLU(inplace=True),
        nn.Dropout(p=dropout_rate2, inplace=False),
        nn.Linear(in_features=num_neurons_in, out_features=2, bias=True),  # Saída final para 2 classes
        nn.Softmax(dim=1)
    )
    model.to(device)

    # Configurar o otimizador
    params_to_update = [p for p in model.parameters() if p.requires_grad]
    if optimizer_name == "SGD":
        optimizer = optim.SGD(params_to_update, lr=lr, momentum=momentum)
    else:
        optimizer = optim.Adam(params_to_update, lr=lr)

    # Definir função de perda
    criterion = nn.CrossEntropyLoss()

    # Ignite trainers
    trainer = Engine(train_step)
    evaluator = Engine(validation_step)

    # Métricas
    val_metrics = {
        "accuracy": Accuracy(),
        "loss": Loss(criterion)
    }
    for name, metric in val_metrics.items():
        metric.attach(evaluator, name)

    @trainer.on(Events.EPOCH_COMPLETED)
    def log_training_results(engine):
        evaluator.run(dataloaders_dict['val'])
        metrics = evaluator.state.metrics
        print(f"Val Accuracy: {metrics['accuracy']:.4f}")
        
        # Early Stopping
        score_function = lambda engine: engine.state.metrics['accuracy']
        handler = EarlyStopping(patience=10, score_function=score_function, trainer=trainer)
        evaluator.add_event_handler(Events.COMPLETED, handler)

    # Executar o treinamento
    trainer.run(dataloaders_dict['train'], max_epochs=100)

    # Obter a acurácia final
    evaluator.run(dataloaders_dict['val'])
    return evaluator.state.metrics["accuracy"]

# Rodar o estudo Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Melhor conjunto de hiperparâmetros
print("Best trial:")
trial = study.best_trial
print(f"Accuracy: {trial.value}")
print("Best hyperparameters: ", trial.params)


with open("result_vgg19.txt", 'a', encoding='utf-8') as file:
    file.write(f'Best trial --- \n Accuracy: {trial.value}\n \n \n')
    file.write(f'Best hyperparameters : {trial.params}')

[I 2025-02-06 23:24:01,523] A new study created in memory with name: no-name-8d1ea76c-c254-41de-ba76-81342982753b
/tmp/ipykernel_138/3588231907.py:29: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  dropout_rate1 = trial.suggest_uniform("dropout1", 0.2, 0.5)
/tmp/ipykernel_138/3588231907.py:30: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  dropout_rate2 = trial.suggest_uniform("dropout2", 0.2, 0.5)
/tmp/ipykernel_138/3588231907.py:36: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-5, 1e-2)
/tmp/ipykernel_138/358

Val Accuracy: 0.5495
Val Accuracy: 0.5495
Val Accuracy: 0.5495
Val Accuracy: 0.5495
Val Accuracy: 0.5495
Val Accuracy: 0.5495
Val Accuracy: 0.5495
Val Accuracy: 0.5495
Val Accuracy: 0.5495
Val Accuracy: 0.5495
Val Accuracy: 0.5495


2025-02-06 23:57:09,292 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Val Accuracy: 0.5495


2025-02-06 23:57:37,946 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-06 23:57:37,947 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
[I 2025-02-06 23:57:37,947] Trial 0 finished with value: 0.5494505494505495 and parameters: {'dropout1': 0.39245854658509727, 'dropout2': 0.2228717421731657, 'activation': 'ReLU', 'batch_size': 128, 'num_neurons_in': 512, 'num_neurons_out': 512, 'optimizer': 'SGD', 'lr': 0.0003250078072195004, 'momentum': 0.8325451987937043}. Best is trial 0 with value: 0.5494505494505495.


...

Val Accuracy: 0.9267
Val Accuracy: 0.9597
Val Accuracy: 0.9634
Val Accuracy: 0.9707
Val Accuracy: 0.9597
Val Accuracy: 0.9634
Val Accuracy: 0.9560
Val Accuracy: 0.9597
Val Accuracy: 0.9670
Val Accuracy: 0.9780
Val Accuracy: 0.9707
Val Accuracy: 0.9780
Val Accuracy: 0.9744
Val Accuracy: 0.9780
Val Accuracy: 0.9780
Val Accuracy: 0.9780
Val Accuracy: 0.9744
Val Accuracy: 0.9670
Val Accuracy: 0.9634


2025-02-08 20:46:09,449 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-08 20:46:09,450 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-08 20:46:09,450 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-08 20:46:09,451 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-08 20:46:09,451 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-08 20:46:09,451 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-08 20:46:09,451 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-08 20:46:09,452 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-08 20:46:09,452 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Val Accuracy: 0.9744


2025-02-08 20:46:38,510 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-08 20:46:38,511 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-08 20:46:38,512 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-08 20:46:38,512 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-08 20:46:38,512 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-08 20:46:38,512 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-08 20:46:38,513 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-08 20:46:38,513 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-08 20:46:38,513 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
[I 2025-02-08 20:46:38,514] Trial 49 finished with value: 0.9743

Best trial:
Accuracy: 0.9853479853479854
Best hyperparameters:  {'dropout1': 0.332301792028777, 'dropout2': 0.2705522735661898, 'activation': 'ReLU', 'batch_size': 128, 'num_neurons_in': 512, 'num_neurons_out': 1024, 'optimizer': 'Adam', 'lr': 0.0013912900114223126}
